# Component 4 garbage model — finish training on Kaggle (free GPU)
Use this when Colab says **"Cannot connect to GPU backend"**.

**Before running:** right panel → *Session options* → **Accelerator: GPU T4 x2** and **Internet: On** (both need a phone-verified Kaggle account). Add your uploaded Kaggle dataset (`c4-training-files`) with **+ Add Input**.

**Best way to run:** click **Save Version → Save & Run All (Commit)**. It runs in the background, even with your laptop closed, for up to 12 hours. The finished model appears in the version's **Output** tab.

## 1. GPU check + install

In [ ]:
!nvidia-smi
!pip -q install ultralytics
import torch
print("CUDA:", torch.cuda.is_available())

## 2. Settings — finds your uploaded files automatically

In [ ]:
import os, glob, shutil, zipfile, yaml
from pathlib import Path

INPUT_ROOT = "/kaggle/input"
MODE = "finetune"        # "finetune" = 25 short epochs from your best.pt (~1.5 h)
                         # "full"     = 60 epochs from yolov8s.pt (~3.5 h)
IMGSZ = 960
CLASSES = ['aluminium foil', 'ball', 'bottle', 'juice', 'leaf', 'thermocol']
USE_SYNTHETIC = True
SYNTH_N = 1500
SYNTH_NEGATIVES = 300
RUNS_DIR = "/kaggle/working/c4_runs"

def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_ROOT}/**/{pattern}", recursive=True))
    return hits[0] if hits else None

yamls = [p for p in glob.glob(f"{INPUT_ROOT}/**/data.yaml", recursive=True)
         if os.path.isdir(os.path.join(os.path.dirname(p), "train"))]
ORIG_DATASET = os.path.dirname(sorted(yamls)[0]) if yamls else None
# Your uploaded best.pt / "best (1).pt" - NOT old runs saved inside the Dataset folder
_weights = [p for p in sorted(glob.glob(f"{INPUT_ROOT}/**/best*.pt", recursive=True))
            if "/runs/" not in p and "/weights/" not in p]
START_WEIGHTS = _weights[0] if _weights else None
SYNTH_SCRIPT = find("synth_pool_data.py")
MY_FRAMES_ZIP = find("my_pool_frames.zip") or "/none"
print("dataset      :", ORIG_DATASET)
print("start weights:", START_WEIGHTS)
print("synth script :", SYNTH_SCRIPT)
print("own frames   :", MY_FRAMES_ZIP)
assert ORIG_DATASET, "Dataset not found: the uploaded files must contain Dataset/data.yaml + Dataset/train"
if MODE == "finetune":
    assert START_WEIGHTS, "best.pt not found in the uploaded files (needed for MODE='finetune')"
SYNTH_SCRIPT = SYNTH_SCRIPT or "/none"

## 3. Merge data (same as the Colab notebook)

In [ ]:
import os, shutil, glob, zipfile, yaml, random
from pathlib import Path

WORK = Path("/tmp/merged")
if WORK.exists(): shutil.rmtree(WORK)
for s in ["train", "valid", "test"]:
    (WORK / s / "images").mkdir(parents=True); (WORK / s / "labels").mkdir(parents=True)

def copy_split(src_root, src_split, dst_split, names_in, prefix):
    """Copies images+labels, re-mapping class ids BY NAME so a different class order can't break labels."""
    remap = {i: CLASSES.index(n) for i, n in enumerate(names_in) if n in CLASSES}
    img_dir = Path(src_root) / src_split / "images"
    n = 0
    for img in glob.glob(str(img_dir / "*")):
        if not img.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")): continue
        stem = Path(img).stem
        lab = Path(src_root) / src_split / "labels" / f"{stem}.txt"
        lines = []
        if lab.exists():
            for line in lab.read_text().splitlines():
                p = line.split()
                if p and int(p[0]) in remap:
                    lines.append(" ".join([str(remap[int(p[0])])] + p[1:]))
        shutil.copy(img, WORK / dst_split / "images" / f"{prefix}{Path(img).name}")
        (WORK / dst_split / "labels" / f"{prefix}{stem}.txt").write_text("\n".join(lines))  # empty file = background image
        n += 1
    return n

orig_names = yaml.safe_load(open(f"{ORIG_DATASET}/data.yaml"))["names"]
for s in ["train", "valid", "test"]:
    print("roboflow", s, copy_split(ORIG_DATASET, s, s, orig_names, "rf_"))

if os.path.exists(MY_FRAMES_ZIP):
    shutil.rmtree("/tmp/my_frames", ignore_errors=True)
    zipfile.ZipFile(MY_FRAMES_ZIP).extractall("/tmp/my_frames")
    my_yaml = glob.glob("/tmp/my_frames/**/data.yaml", recursive=True)[0]
    my_root = str(Path(my_yaml).parent)
    my_names = yaml.safe_load(open(my_yaml))["names"]
    for s in ["train", "valid", "test"]:
        if (Path(my_root) / s / "images").exists():
            print("my camera", s, copy_split(my_root, s, s, my_names, "my_"))
else:
    print("No own-camera frames found - training on Roboflow only (accuracy on YOUR camera will stay limited).")

## 4. Leakage check

In [ ]:
def origin(name):  # "1013_jpg.rf.abc.jpg" -> "rf_1013"
    base = Path(name).stem
    return base.split("_jpg.rf.")[0].split(".rf.")[0]

held_out = {origin(p) for s in ["valid", "test"] for p in os.listdir(WORK/s/"images")}
removed = 0
for img in list(os.listdir(WORK/"train/images")):
    if origin(img) in held_out:
        os.remove(WORK/"train/images"/img)
        lab = WORK/"train/labels"/f"{Path(img).stem}.txt"
        if lab.exists(): os.remove(lab)
        removed += 1
print("leaked copies removed from train:", removed)
for s in ["train", "valid", "test"]:
    imgs = os.listdir(WORK/s/"images")
    empty = sum(1 for p in os.listdir(WORK/s/"labels") if os.path.getsize(WORK/s/"labels"/p) == 0)
    mine = sum(1 for p in imgs if p.startswith("my_"))
    print(f"{s:6s} images={len(imgs):5d}  own-camera={mine:4d}  background(empty)={empty}")

## 5. Synthetic images

In [ ]:
import subprocess, sys
if USE_SYNTHETIC and os.path.exists(SYNTH_SCRIPT):
    yaml.safe_dump({"names": CLASSES, "nc": len(CLASSES)}, open(WORK / "data.yaml", "w"))
    subprocess.run([sys.executable, SYNTH_SCRIPT, "--data", str(WORK), "--out", "/tmp/synth",
                    "--n", str(SYNTH_N), "--negatives", str(SYNTH_NEGATIVES)], check=True)
    for sub in ["images", "labels"]:
        for f in os.listdir(f"/tmp/synth/{sub}"):
            shutil.move(f"/tmp/synth/{sub}/{f}", WORK / "train" / sub / f)
    print("train images now:", len(os.listdir(WORK / "train/images")))
else:
    print("Synthetic step skipped (USE_SYNTHETIC is False or synth_pool_data.py is not in MyDrive).")

## 6. data.yaml

In [ ]:
data_yaml = WORK / "data.yaml"
yaml.safe_dump({"path": str(WORK), "train": "train/images", "val": "valid/images",
                "test": "test/images", "nc": len(CLASSES), "names": CLASSES},
               open(data_yaml, "w"), sort_keys=False)
print(open(data_yaml).read())

## 7. Train

In [ ]:
from ultralytics import YOLO
common = dict(data=str(data_yaml), imgsz=IMGSZ, batch=-1, project=RUNS_DIR, exist_ok=True,
              hsv_h=0.02, hsv_s=0.7, hsv_v=0.5, degrees=8, translate=0.15, scale=0.6,
              fliplr=0.5, cache="disk", workers=4, seed=0, device=0)
if MODE == "finetune":
    YOLO(START_WEIGHTS).train(name="garbage_v2_ft", epochs=25, patience=25, lr0=0.001, lrf=0.1,
                              warmup_epochs=0, cos_lr=True, close_mosaic=10, mixup=0.0,
                              save_period=5, **common)
    RUN = "garbage_v2_ft"
else:
    YOLO("yolov8s.pt").train(name="garbage_v2", epochs=60, patience=60, cos_lr=True,
                             close_mosaic=15, shear=2.0, perspective=0.0005, mixup=0.1,
                             save_period=10, **common)
    RUN = "garbage_v2"

## 8. Evaluate + save the model to Output

In [ ]:
best = f"{RUNS_DIR}/{RUN}/weights/best.pt"
m = YOLO(best)
for split in ["val", "test"]:
    r = m.val(data=str(data_yaml), split=split, imgsz=IMGSZ, name=f"eval_{split}")
    print(f"{split}: P={r.box.mp:.3f} R={r.box.mr:.3f} mAP50={r.box.map50:.3f} mAP50-95={r.box.map:.3f}")
if START_WEIGHTS:
    r0 = YOLO(START_WEIGHTS).val(data=str(data_yaml), split="test", imgsz=IMGSZ, name="eval_start")
    print(f"start model test: mAP50={r0.box.map50:.3f} mAP50-95={r0.box.map:.3f}")
shutil.copy(best, "/kaggle/working/swimming_pool_garbage_yolo_v2.pt")
shutil.rmtree("/tmp/merged", ignore_errors=True)
print("Download /kaggle/working/swimming_pool_garbage_yolo_v2.pt from the Output tab.")